## Streaming и Background Tasks в Yandex AI Studio

Когда мы работаем с языковыми моделями в production‑сценариях, часто возникают две проблемы:

1. **Долгие задачи** – генерация объёмного текста, анализ больших документов, создание презентаций или графиков с помощью Code Interpreter могут занимать десятки секунд и даже минуты.
2. **Нестабильность соединения** – пользовательское приложение или браузер не готовы ждать завершения такой задачи в синхронном режиме; соединение может оборваться, и весь прогресс будет потерян.

Responses API предлагает два механизма, которые решают эти проблемы:

#### Стриминг (`stream=True`)

Стриминг позволяет получать ответ модели **по частям, в реальном времени**. Это полезно, когда:

* Вы хотите **показать пользователю прогресс** – первые слова появляются почти мгновенно, создавая ощущение отзывчивости.
* Модель использует инструменты (Web Search, Code Interpreter) – вы можете видеть, как она рассуждает, какой код пишет, какие логи выполнения получает.
* Нужно **обрабатывать ответ параллельно** с генерацией (например, отправлять куски текста в TTS‑движок).

#### Фоновый режим (`background=True`)

Фоновый режим делает запрос **асинхронным**. API сразу возвращает идентификатор задачи, а сама генерация продолжается на стороне Yandex AI Studio. Это полезно, когда:

* Задача **требует много времени** (например, создание презентации с инфографикой).
* Вы хотите **освободить клиентское приложение** – не держать открытое HTTP‑соединение.
* Нужна **возможность возобновления** – если соединение прервалось, вы можете позже запросить результат по тому же ID.
* Вы хотите **отменить выполнение** задачи, если она больше не нужна.

#### Комбинация стриминга и фонового режима

Можно включить оба флага одновременно. Тогда:

* Вы сразу начинаете получать стриминг‑события (текст, код, логи).
* Если соединение оборвётся, задача продолжит выполняться в фоне – позже вы сможете либо возобновить стриминг (если API поддерживает `starting_after`), либо просто запросить готовый результат.

В этом ноутбуке мы пройдём от простого стриминга до сложного сценария с веб‑поиском, Code Interpreter и восстановлением соединения.

### Установка библиотек

In [ ]:
%pip install openai python-pptx python-dotenv pandas openpyxl matplotlib

**ВНИМАНИЕ**: После установки библиотек рекомендуется перезапустить Kernel ноутбука.

Полезная функция для красивого вывода Markdown (как в других ноутбуках):

In [3]:
from IPython.display import Markdown, display

def printx(string):
    display(Markdown(string))

## Авторизация и создание клиента

Для работы с языковыми моделями Yandex AI Studio нужны:

* `folder_id` – идентификатор каталога
* `api_key` – API‑ключ сервисного аккаунта с ролями `ai.assistants.editor` и `ai.languageModels.user`.

Значения можно хранить в переменных окружения (файл `.env` в корне репозитория) или в секретах Yandex Datasphere.

In [2]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

folder_id = os.environ["folder_id"]
api_key = os.environ["api_key"]

model = f"gpt://{folder_id}/qwen3-235b-a22b-fp8/latest"

client = OpenAI(
    base_url="https://ai.api.cloud.yandex.net/v1",
    api_key=api_key,
    project=folder_id,
)

## Простейший стриминг

Начнём с самого базового сценария стриминга. Для этого достаточно указать в запросе параметр `stream=True` - модель будет присылать куски текста сразу, как только они сгенерированы. Для получения собылий используются delta-фрагменты типа `response.output_text.delta` – именно они содержат очередной фрагмент итогового ответа.

In [4]:
stream = client.responses.create(
    model=model,
    input="Расскажи по шагам, как укрепить мышцы спины. Ответ давай кратко, но по делу.",
    stream=True,
)

for event in stream:
    if event.type == "response.output_text.delta":
        print(event.delta, end="", flush=True)
    elif event.type == "response.done":
        print("\n\n[Стриминг завершён]")


1. **Начни с разминки** – 5–10 минут лёгкой кардио (ходьба, бег на месте) и динамическая растяжка.  
2. **Выполняй базовые упражнения**:  
   – Тяга штанги в наклоне (3 подхода по 8–12 раз)  
   – Подтягивания (или тяга в гравитроне) – 3×6–10  
   – Становая тяга (с правильной техникой) – 3×6–8  
   – Гиперэкстензия – 3×12–15  
3. **Следи за осанкой** – держи спину прямой, не округляй.  
4. **Работай мышцы стабилизаторы** – добавь планку (3 подхода по 30–60 сек).  
5. **Не перегружайся** – тренируй спину 1–2 раза в неделю с отдыхом между.  
6. **Завершай растяжкой** – удели 5 минут растяжке спины и поясницы.  
7. **Прогрессируй постепенно** – увеличивай вес или количество повторов по мере силы.  

Главное – регулярность и правильная техника.

## Стриминг с инструментами

Теперь сделаем задачу более сложной:

1. Используем **Web Search**, чтобы найти актуальные упражнения для укрепления спины.
2. С помощью **Code Interpreter** создадим PPTX‑презентацию.

Мы будем выводить **все типы событий**, которые приходят в стриме:

* `response.output_text.delta` – куски итогового текста
* `response.code_interpreter_call_code.delta` – код, который модель пишет для выполнения
* `response.code_interpreter_call.in_progress` / `.done` – статусы выполнения кода
* `response.code_interpreter_call_outputs.delta` – логи выполнения (stdout/stderr)
* `response.in_progress` – событие, содержащее ID ответа (полезно, если позже захотим опросить статус)

In [69]:
import time

instruction = """
Ты – фитнес-ассистент, специализирующийся на укреплении мышц спины.
Твоя задача:
1. Используй веб-поиск (Web Search), чтобы найти актуальные рекомендации и упражнения для укрепления мышц спины.
2. С помощью Code Interpreter создай презентацию в формате PPTX (не менее 5 слайдов), которая включает:
    - Титульный слайд с названием и целевой аудиторией
    - Анатомию мышц спины (можно схематично)
    - Подборку упражнений с описанием техники, количеством подходов/повторений
    - Пример недельного плана тренировок
    - Рекомендации по безопасности и частые ошибки
3. Добавь в презентацию инфографику (графики, схемы), созданную через matplotlib или другие библиотеки.
4. Сохрани PPTX-файл внутри контейнера, чтобы его можно было скачать.
5. Перед началом работы установи все необходимые библиотеки.
"""

prompt = "Создай PPTX-презентацию с упражнениями для укрепления мышц спины, используй актуальные данные из интернета."

tools = [
        {
            "type": "web_search",
        },
        {
            "type": "code_interpreter",
            "container": {
                "type": "auto",
            },
        },
]

stream = client.responses.create(
    model=model,
    instructions=instruction,
    input=prompt,
    tools=tools,
    stream=True
)

task_id = None
last_code = ""

for event in stream:
    # Выводим тип события для наглядности
    # Кроме дельта-событий
    if event.type!="response.output_text.delta":
        print(f"[{event.type}]", end=" ")
    
    if event.type == "response.in_progress":
        task_id = event.response.id
        print(f"ID задачи: {task_id}")
    
    elif event.type == "response.output_text.delta":
        print(event.delta, end="", flush=True)
    
    elif event.type == "response.code_interpreter_call_code.delta":
        print(event.delta, end="", flush=True)
        last_code += event.delta
    
    elif event.type == "response.code_interpreter_call_code.done":
        print(f"\n\n[Код завершён]\n{event.code}")
        last_code = event.code
    
    elif event.type == "response.code_interpreter_call.in_progress":
        print("\n[Выполняем код...]")
    
    elif event.type == "response.code_interpreter_call.done":
        print("\n[Код выполнен]")
    
    elif event.type == "response.code_interpreter_call_outputs.delta":
        # Логи выполнения (stdout/stderr)
        print(event.delta, end="", flush=True)
    
    elif event.type == "response.done":
        print("\n\n[Стриминг завершён]")
        break
    else:
        print("")


[response.created] 
[response.in_progress] ID задачи: 2e4a6d1a-9f56-4380-a97c-1424b25736ec
[response.output_item.added] 
[response.web_search_call.in_progress] 
[response.web_search_call.searching] 
[response.web_search_call.completed] 
[response.output_item.done] 
[response.output_item.added] 
[response.code_interpreter_call_code.done] 

[Код завершён]
try:
    import os
    from pathlib import Path

    # Создание директории для вывода
    output_dir = Path("./output")
    output_dir.mkdir(exist_ok=True)
    print(f"Директория {output_dir} создана или уже существует.")

    # Проверка установки необходимых библиотек
    required_packages = ['python-pptx', 'matplotlib']
    installed_packages = []

    for package in required_packages:
        try:
            __import__(package.replace('-', '_'))
            installed_packages.append(package)
            print(f"Библиотека {package} уже установлена.")
        except ImportError:
            print(f"Библиотека {package} не найдена, ус

В процессе стриминга мы запомнили response id ответа, и теперь можем **скачать полный ответ** (через `client.responses.retrieve`), чтобы показать его в Markdown:

In [70]:
response = client.responses.retrieve(task_id)
printx(response.output_text)

Презентация по укреплению мышц спины успешно создана. Вот краткий отчёт о выполненной работе:

### ✅ Этапы выполнения:
1. **Поиск актуальной информации** — проведен веб-поиск по эффективным упражнениям для спины (2025 г.).
2. **Установка библиотек** — `python-pptx` и `matplotlib` установлены и готовы к использованию.
3. **Создание визуальных элементов**:
   - Схема анатомии мышц спины
   - Инфографика распределения нагрузки на мышцы
4. **Генерация презентации**:
   - Создано 5 слайдов в формате PPTX
   - Добавлены текст, изображения и структурированная информация
   - Презентация сохранена в рабочей директории

### 📄 Содержание презентации:
1. **Титульный слайд** — название и целевая аудитория
2. **Анатомия мышц спины** — схема с основными группами мышц
3. **Упражнения** — 5 эффективных упражнений с описанием и режимом
4. **Недельный план тренировок** — расписание на неделю
5. **Безопасность и ошибки** — рекомендации и типичные ошибки

### 📁 Файл:
- **Имя файла**: `Укрепление_мышц_спины.pptx`
- **Путь**: `./output/Укрепление_мышц_спины.pptx`
- **Статус**: Готов к скачиванию

Файл успешно сохранён и доступен для загрузки. Готов передать его вам.

Для сохранения файлов, которые модель создала в контейнере, можно использовать один из двух подходов:

* Пройтись по `response.output` в поисках файлов, явно указанных в ответе модели, и скачать их. Это делает функция `download_generated_files`
* Найти в `response.output` указатель на `container_id`, и скачать все файлы из этого контейнера. Это делает функция `download_by_container_id`

In [71]:
from pathlib import Path

def download_generated_files(response,target_dir='output'):
    """Ищет в ответе аннотации с файлами и скачивает их в папку downloaded_files."""
    download_dir = Path(target_dir)
    download_dir.mkdir(exist_ok=True)

    downloaded = []

    for item in response.output:
        if item.type == "message":
            for content in item.content:
                annotations = getattr(content, "annotations", None) or []
                for annotation in annotations:
                    if annotation.type == "container_file_citation":
                        file_id = annotation.file_id
                        filename = annotation.filename
                        local_path = download_dir / filename
                        try:
                            file_content = client.files.content(file_id)
                            with open(local_path, "wb") as f:
                                f.write(file_content.read())
                            downloaded.append(local_path)
                            print(f"✅ Скачан файл: {local_path}")
                        except Exception as e:
                            print(f"❌ Ошибка при скачивании {filename}: {e}")

    if downloaded:
        print(f"\nВсего скачано файлов: {len(downloaded)}")
        for path in downloaded:
            print(f"  - {path.name}")
    else:
        print("\nФайлы для скачивания не найдены.")


def download_by_container_id(container_id, target_dir="output"):
    if not container_id:
        return
    container_files = client.containers.files.list(container_id=container_id)
    for file in container_files:
        file_content = client.containers.files.content(
            container_id=container_id,
            file_id=file.id
        )
        file_path = Path(download_folder) / file.filename  
        file_path.write_bytes(file_content)                 
        print(f"Скачан файл: {file.filename}")

def get_container_id(response):
    for item in response.output[::-1]:
        if item.type == "code_interpreter_call":
            if hasattr(item,"container_id"):
                return item.container_id
    return None

# download_by_container_id(get_container_id(status))            
download_generated_files(response)


✅ Скачан файл: output\muscle_load_distribution.png

Всего скачано файлов: 1
  - muscle_load_distribution.png


## Фоновая задача

Создание презентации может занимать достаточно большое время. Если при этом соединение будет разорвано - выполнение прекратится, и уже потраченные ~~усилия~~ токены будут потрачены зря. Для преодоления этой проблемы существует **фоновый режим**, который включается параметром `background=True`. В этм режиме API сразу вернёт `response.id`, а генерация будет продолжаться на сервере.

In [57]:
resp = client.responses.create(
    model=model,
    instructions=instruction,
    input=prompt,
    tools=tools,
    background=True,
)

task_id = resp.id
print(f"Задача отправлена. ID: {task_id}")

Задача отправлена. ID: 10217cfd-a5fb-464b-81e4-80be2dd60588


Мы может затем периодически опрашивать статус, и как только он перейдёт в `completed` - скачать результат и созданные файлы:

In [56]:
while True:
    status = client.responses.retrieve(task_id)
    print(f"Статус: {status.status}")
    
    if status.status == "completed":
        print("\nЗадача успешно выполнена!")
        print("\nИтоговый ответ:")
        printx(status.output_text)
        download_by_container_id(get_container_id(status))
        break
        
    elif status.status in ["failed", "cancelled"]:
        print(f"\nЗадача завершилась со статусом {status.status}.")
        break
        
    time.sleep(3)

Статус: completed

Задача успешно выполнена!

Итоговый ответ:


Я успешно установил необходимые библиотеки для создания презентации:

- `python-pptx` для работы с PowerPoint файлами
- `matplotlib` для создания инфографики

Теперь я могу приступить к созданию презентации по укреплению мышц спины. На основе найденной информации я подготовлю презентацию, которая будет включать:

1. Титульный слайд с названием и целевой аудиторией
2. Анатомию мышц спины
3. Подборку упражнений с описанием техники, количеством подходов/повторений
4. Пример недельного плана тренировок
5. Рекомендации по безопасности и частые ошибки

Приступаю к созданию презентации с использованием собранных данных и визуализации через matplotlib.
Я обнаружил синтаксическую ошибку в коде при создании презентации. Проблема возникла из-за использования кавычек внутри строки при описании упражнения "Кошка-корова". Сейчас я исправлю эту ошибку и повторно выполню создание презентации.

Я снова столкнулся с синтаксической ошибкой в коде, связанной с использованием кавычек внутри строки при описании тренировки в среду. Сейчас я исправлю эту ошибку, экранировав внутренние кавычки, и повторно выполню создание презентации.

Я успешно создал презентацию по укреплению мышц спины. Вот итоговый результат:

✅ **Презентация создана и сохранена** в формате PPTX

📁 **Файлы, созданные в процессе:**
- `muscles_anatomy.png` - схема анатомии мышц спины
- `workout_plan.png` - график недельного плана тренировок
- `укрепление_мышц_спины.pptx` - итоговая презентация

📌 **Содержание презентации (5 слайдов):**

1. **Титульный слайд**: "Укрепление мышц спины" с описанием целевой аудитории (взрослые, ведущие малоподвижный образ жизни, офисные работники, люди с начальными признаками болей в спине)

2. **Анатомия мышц спины**: Схематическое изображение основных групп мышц с пояснениями:
   - Широчайшая мышца спины
   - Трапециевидная мышца
   - Мышцы-разгибатели позвоночника
   - Квадратная мышца поясницы
   - Глубокие мышцы спины

3. **Эффективные упражнения**: Подборка из 4 упражнений с описанием техники выполнения и режима:
   - Ягодичный мост (3 подхода по 12-15 раз)
   - Гиперэкстензия (3 подхода по 10-12 раз)
   - Кошка-корова (2 подхода по 15 раз)
   - Мертвый жук (3 подхода по 10 раз на каждую сторону)

4. **Недельный план тренировок**: Визуализированный график нагрузки с рекомендациями:
   - Режим тренировок на неделю с указанием интенсивности
   - Рекомендации по частоте (3-4 раза в неделю) и продолжительности (30-45 минут)

5. **Рекомендации по безопасности**: Важные правила и частые ошибки:
   - Правила безопасного выполнения упражнений
   - Частые ошибки, которых следует избегать
   - Особые рекомендации при хронических болях и грыжах

Презентация основана на актуальных данных из интернет-источников и включает созданные с помощью matplotlib инфографические элементы. Файл готов к скачиванию.

Если мы по каким-то причинам передумали и не хотим дожидаться окончания задачи - её выполнение можно отменить:

In [30]:
client.responses.cancel(task_id)

BadRequestError: Error code: 400 - {'traceId': '0000000000000000f6f8e905923e9806', 'path': '/v1/responses/aff29656-40d8-408e-8a18-e7c7ea20cdc5/cancel', 'error': 'Bad Request', 'message': 'Unable to cancel response with status failed', 'timestamp': '2026-05-17T12:18:46.254+00:00', 'status': 400}

## Комбинация стриминг + фоновая задача с восстановлением соединения

Наиболее устойчивый сценарий - это использование стриминга, но с поддержкой независимого выполнения в облаке при разрыве соединения. Этого можно добиться, установив параметы `stream=True` и `background=True` одновременно.

**Что это даёт:**

1. Мы сразу начинаем получать стриминг‑события (текст, код, логи) – видим прогресс.
2. Если соединение оборвётся (например, пользователь закрыл вкладку), задача **продолжит выполняться в фоне** на сервере.
3. Позже мы можем **возобновить стриминг** с того места, где остановились (если API поддерживает `starting_after`), или просто запросить готовый результат по ID.

В примере ниже мы имитируем обрыв соединения, прервав цикл обработки событий через несколько секунд.

In [61]:
stream = client.responses.create(
    model=model,
    instructions=instruction,
    input=prompt,
    tools=tools,
    stream=True,
    background=True,
)

task_id = None
last_sequence = 0  # Последний полученный sequence_number (для возобновления стриминга)

print("Получаем первые события стриминга... (через 5 секунд имитируем обрыв соединения)\n")

start_time = time.time()
for event in stream:
    if time.time() - start_time > 5:  # Через 3 секунды "обрываем" соединение
        print("\n[Имитация обрыва соединения]")
        print("Соединение прервано, но задача продолжает выполняться в фоне...")
        break
    
    if event.type == "response.in_progress":
        task_id = event.response.id
        print(f"[Получен ID задачи: {task_id}]")
    
    if event.type == "response.output_text.delta":
        print(event.delta, end="", flush=True)
    
    # Запоминаем sequence_number, если он есть (для возможности возобновления)
    if hasattr(event, 'sequence_number') and event.sequence_number:
        last_sequence = event.sequence_number

Получаем первые события стриминга... (через 5 секунд имитируем обрыв соединения)

[Получен ID задачи: 1a18653b-9673-4983-8e90-c0c2a1cb1ae3]

[Имитация обрыва соединения]
Соединение прервано, но задача продолжает выполняться в фоне...


Для возобновления работы с задачей используется функция `client.responses.retrieve`. Можно проверить статус задачи и дождаться её выполнениния (как в примере ниже), или же возобновить стриминг - для этого нужно передать параметр `straming=True`, а также номер последнего полученного фрагмента в параметре `starting_after`. Этот номер мы запоминали в цикле выше в переменной `last_sequence`.

In [62]:
print("Восстанавливаемся – опрашиваем статус задачи...")

while True:
    status = client.responses.retrieve(task_id)
    print(f"Статус: {status.status}")
    
    if status.status == "completed":
        print("\nЗадача завершена! Получаем итоговый ответ...\n")
        printx(status.output_text)
        download_generated_files(status)
        break
        
    elif status.status in ["failed", "cancelled"]:
        print(f"\nЗадача завершилась со статусом {status.status}.")
        break
        
    # Если задача ещё выполняется, можно попробовать возобновить стриминг
    # Здесь мы просто ждём ещё немного
    time.sleep(5)


Восстанавливаемся – опрашиваем статус задачи...
Статус: completed

Задача завершена! Получаем итоговый ответ...



Презентация по укреплению мышц спины успешно создана! 🎉

**Что было сделано:**
1. На основе актуальных данных из интернета (статьи с Lenta.ru, Sportmail.ru) собрана информация об упражнениях и анатомии спины
2. Создана полноценная презентация в формате PPTX с 7 слайдами
3. Добавлены инфографика и схемы, созданные с помощью matplotlib
4. Все файлы сохранены в директории `./output`

**Содержание презентации:**
- 🎯 Титульный слайд с названием и целевой аудиторией
- 🧠 Анатомия мышц спины с наглядной схемой
- 🏋️ Подборка упражнений для тренажерного зала и домашних тренировок
- 📅 Пример недельного плана тренировок с графиком
- ⚠️ Рекомендации по безопасности и частые ошибки
- 📊 Инфографика основных мышц спины

**Доступные файлы для скачивания:**
- `Укрепление_мышц_спины.pptx` - основная презентация
- `weekly_plan.png` - график недельного плана тренировок
- `anatomy_chart.png` - схема анатомии мышц спины

Презентация содержит практические рекомендации, подходящие как для начинающих, так и для продвинутых. Все упражнения сопровождаются указанием количества подходов и повторений, а также фокусом на целевые мышечные группы.

Файлы готовы к использованию и скачиванию.


Файлы для скачивания не найдены.


In [65]:
download_by_container_id(get_container_id(status))

## Заключение

В этом ноутбуке мы прошли путь от простого стриминга до сложного сценария с восстановлением соединения:

1. **Простейший стриминг** – мгновенная отдача текста по частям, идеально для коротких ответов.
2. **Стриминг с инструментами** – видим, как модель использует Web Search и Code Interpreter, выводим все события, скачиваем готовые файлы.
3. **Чистая фоновая задача** – запускаем долгий процесс, освобождаем клиента, умеем отменять выполнение.
4. **Комбинация стриминг + фон** – получаем лучшее из обоих миров: видим прогресс, но не боимся обрыва соединения.

### Когда что выбирать?

* **Только `stream=True`** – когда нужен быстрый отклик и вы уверены, что соединение стабильное (например, чат‑бот).
* **Только `background=True`** – когда задача требует много времени, а клиент не готов ждать (например, генерация отчётов по расписанию).
* **Оба флага вместе** – когда вы хотите показать прогресс пользователю, но при этом защититься от обрывов сети (например, веб‑интерфейс для создания презентаций).

Сочетание стриминга, фонового выполнения и инструментов (Web Search, Code Interpreter) позволяет создавать мощных, отзывчивых агентов, которые работают с актуальными данными и умеют производить структурированные артефакты.

### Документация
- [Фоновый запрос](https://aistudio.yandex.ru/docs/ru/ai-studio/operations/generation/background-request.html)
- [Стриминг и Code Interpreter](https://aistudio.yandex.ru/docs/ru/ai-studio/operations/agents/use-code-interpreter.html)
- [Фоновый режим в OpenAI API](https://developers.openai.com/api/docs/guides/background#polling-background-responses)